# Tiền xử lý bước 1

## Bước này tạo file:

- df_inter.label.parquet.

File này chia dữ liệu theo x_label có 3 giá trị: 0, 1, 2 tương ứng với train, val, test.


# Train/Validation/Test data splitting

- Based on generated interactions, perform data splitting


In [2]:
import os
import yaml
import pandas as pd

In [3]:
PATH = "./data/2014"

## Load interactions


In [5]:
df = pd.read_parquet(os.path.join(PATH, "df_inter.parquet"))

In [7]:
print(f"shape: {df.shape}")
df[:4]

shape: (160189, 6)


,userID,itemID,rating,timestamp,reviewerID,asin
0,0,0,5.0,1373932800,A1HK2FQW6KXQB2,097293751X
1,1,0,5.0,1372464000,A19K65VY14D13R,097293751X
2,2,0,5.0,1395187200,A2LL1TGG90977E,097293751X
3,3,0,5.0,1376697600,A5G19RYX8599E,097293751X


Đoạn code này thực hiện hai thao tác xử lý dữ liệu có vẻ trái ngược nhau nhưng lại rất phổ biến trong quá trình chuẩn bị dữ liệu cho hệ thống gợi ý.


In [22]:
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df.sort_values(by=["userID", "timestamp"], inplace=True)

df[:10]

,userID,itemID,rating,timestamp,reviewerID,asin,x_label
19087,0,3858,4.0,1357689600,A1HK2FQW6KXQB2,B004071ZOY,0
92215,0,1915,5.0,1363996800,A1HK2FQW6KXQB2,B001F8TLLU,0
27584,0,0,5.0,1373932800,A1HK2FQW6KXQB2,097293751X,2
66202,0,1580,5.0,1373932800,A1HK2FQW6KXQB2,B0013FGWD0,1
119860,0,1872,5.0,1373932800,A1HK2FQW6KXQB2,B001DKHPD6,0
147905,1,5381,5.0,1367452800,A19K65VY14D13R,B006OK476S,0
96654,1,1427,5.0,1372377600,A19K65VY14D13R,B000YDIGCC,0
76270,1,5159,5.0,1372464000,A19K65VY14D13R,B005SPCXKC,0
91960,1,0,5.0,1372464000,A19K65VY14D13R,097293751X,0
121849,1,2818,5.0,1372464000,A19K65VY14D13R,B002QBDMDI,1


In [23]:
# 1. Khai báo tên cột
uid_field, iid_field = "userID", "itemID"

# 2. Nhóm dữ liệu (Groupby)
uid_freq = df.groupby(uid_field)[iid_field]
u_i_dict = {}
for u, u_ls in uid_freq:
    u_i_dict[u] = list(u_ls)
list(u_i_dict.items())[:3]

[(0, [3858, 1915, 0, 1580, 1872]),
 (1, [5381, 1427, 5159, 0, 2818, 1174]),
 (2, [3999, 6556, 2836, 0, 6712])]

In [11]:
list(u_i_dict.keys())[:3]

[0, 1, 2]

In [13]:
new_label = []
u_ids_sorted = sorted(u_i_dict.keys())
for u in u_ids_sorted:
    items = u_i_dict[u]
    # get num interact
    n_items = len(items)
    if n_items < 10:
        # take 1 for test 1 for val and rest for train
        tmp_ls = [0] * (n_items - 2) + [1] + [2]
    else:
        # split 80% train, 10% val, 10% test
        val_test_len = int(n_items * 0.2)
        train_len = n_items - val_test_len
        val_len = val_test_len // 2
        test_len = val_test_len - val_len
        tmp_ls = [0] * train_len + [1] * val_len + [2] * test_len
    new_label.extend(tmp_ls)

new_label[:10]

[0, 0, 0, 1, 2, 0, 0, 0, 0, 1]

In [17]:
df["x_label"] = new_label
df[:20]

,userID,itemID,rating,timestamp,reviewerID,asin,x_label
19087,0,3858,4.0,1357689600,A1HK2FQW6KXQB2,B004071ZOY,0
92215,0,1915,5.0,1363996800,A1HK2FQW6KXQB2,B001F8TLLU,0
27584,0,1872,5.0,1373932800,A1HK2FQW6KXQB2,B001DKHPD6,0
66202,0,1580,5.0,1373932800,A1HK2FQW6KXQB2,B0013FGWD0,1
119860,0,0,5.0,1373932800,A1HK2FQW6KXQB2,097293751X,2
147905,1,5381,5.0,1367452800,A19K65VY14D13R,B006OK476S,0
96654,1,1427,5.0,1372377600,A19K65VY14D13R,B000YDIGCC,0
76270,1,0,5.0,1372464000,A19K65VY14D13R,097293751X,0
91960,1,5159,5.0,1372464000,A19K65VY14D13R,B005SPCXKC,0
121849,1,2818,5.0,1372464000,A19K65VY14D13R,B002QBDMDI,1


In [26]:
df.to_parquet(os.path.join(PATH, "df_inter.label.parquet"), index=False)

## Reload


In [5]:
indexed_df = pd.read_parquet(os.path.join(PATH, "df_inter.label.parquet"))
print(f"shape: {indexed_df.shape}")
indexed_df[:20]

shape: (160189, 7)


,userID,itemID,rating,timestamp,reviewerID,asin,x_label
0,0,3858,4.0,1357689600,A1HK2FQW6KXQB2,B004071ZOY,0
1,0,1915,5.0,1363996800,A1HK2FQW6KXQB2,B001F8TLLU,0
2,0,0,5.0,1373932800,A1HK2FQW6KXQB2,097293751X,2
3,0,1580,5.0,1373932800,A1HK2FQW6KXQB2,B0013FGWD0,1
4,0,1872,5.0,1373932800,A1HK2FQW6KXQB2,B001DKHPD6,0
5,1,5381,5.0,1367452800,A19K65VY14D13R,B006OK476S,0
6,1,1427,5.0,1372377600,A19K65VY14D13R,B000YDIGCC,0
7,1,5159,5.0,1372464000,A19K65VY14D13R,B005SPCXKC,0
8,1,0,5.0,1372464000,A19K65VY14D13R,097293751X,0
9,1,2818,5.0,1372464000,A19K65VY14D13R,B002QBDMDI,1


In [14]:
train_df = indexed_df[["userID", "itemID", "rating", "timestamp", "x_label"]].copy()
train_df

,userID,itemID,rating,timestamp,x_label
0,0,3858,4.0,1357689600,0
1,0,1915,5.0,1363996800,0
2,0,0,5.0,1373932800,2
3,0,1580,5.0,1373932800,1
4,0,1872,5.0,1373932800,0
...,...,...,...,...,...
160184,19371,6969,5.0,1393632000,0
160185,19371,6997,5.0,1393804800,0
160186,19371,6935,5.0,1394064000,0
160187,19371,6998,4.0,1394668800,1


In [ ]:
# Lưu lại file .inter để train, tùy dataset mà tên bỏ vào thư mục data sẽ khác
# ví dụ data/baby/baby.inter, data/beauty/beauty.inter, data/clothing/clothing.inter
train_df.to_csv(os.path.join(PATH, "df_inter.label.inter"), sep="\t")

In [25]:
u_id_str, i_id_str = "userID", "itemID"
u_uni = indexed_df[u_id_str].unique()
c_uni = indexed_df[i_id_str].unique()

print(f"# of unique learners: {len(u_uni)}")
print(f"# of unique courses: {len(c_uni)}")

print("min/max of unique learners: {0}/{1}".format(min(u_uni), max(u_uni)))
print("min/max of unique courses: {0}/{1}".format(min(c_uni), max(c_uni)))

# of unique learners: 19372
# of unique courses: 7025
min/max of unique learners: 0/19371
min/max of unique courses: 0/7024
